# Sync to Overleaf

Run the cell and enter your **Overleaf Git token** at the hidden prompt. Current local files, including uncommitted edits, replace the Overleaf files. Git-ignored files are skipped. Local files and Git history stay unchanged; the token is not saved.

Requires Python 3.8+ and Git on macOS or Linux. Open this notebook from the thesis repository.


In [ ]:
import base64
import getpass
import os
from pathlib import Path
import re
import shlex
import shutil
import subprocess
import sys
import tempfile
import warnings

def sync_overleaf():
    url = "https://git@git.overleaf.com/670cb15e059a397a1f6cfad1"
    env = {k: v for k, v in os.environ.items() if not k.startswith("GIT_")}
    env.update(GIT_TERMINAL_PROMPT="0", GIT_OPTIONAL_LOCKS="0")
    token = ""
    with tempfile.TemporaryDirectory(prefix="overleaf-sync-") as folder:
        helper = Path(folder) / "credential.py"
        helper.write_text(
            "import os, sys\n"
            "if sys.argv[-1] == 'get':\n"
            "    token = os.read(int(sys.argv[1]), 4096).decode()\n"
            "    sys.stdout.write('username=git\\npassword=' + token + '\\n\\n')\n",
            encoding="utf-8",
        )

        def git(*args, cwd=None, authenticate=False):
            command = ["git", "--no-optional-locks"]
            descriptor = None
            try:
                if authenticate:
                    descriptor, writer = os.pipe()
                    try:
                        os.write(writer, token.encode("utf-8"))
                    finally:
                        os.close(writer)
                    credential = "!{} {} {}".format(
                        shlex.quote(sys.executable), shlex.quote(str(helper)), descriptor
                    )
                    command += ["-c", "credential.helper=", "-c",
                                "credential.helper=" + credential,
                                "-c", "http.followRedirects=false",
                                "-c", "http.version=HTTP/1.1"]
                    if sys.platform == "darwin":
                        command += ["-c", "http.sslBackend=secure-transport"]
                return subprocess.run(
                    command + list(args), cwd=cwd, env=env, check=True,
                    stdout=subprocess.PIPE, stderr=subprocess.PIPE, timeout=1200,
                    pass_fds=() if descriptor is None else (descriptor,),
                ).stdout
            except subprocess.CalledProcessError as error:
                detail = error.stderr.decode(errors="replace").strip()
                if token:
                    encoded = base64.b64encode(("git:" + token).encode()).decode()
                    detail = detail.replace(token, "[redacted]").replace(encoded, "[redacted]")
                detail = re.sub(r"(https?://)[^/\s@]+@", r"\1[redacted]@", detail)
                detail = re.sub(r"(?im)^.*(?:authorization|password)\s*[:=].*$", "[redacted]", detail)
                operation = next((a for a in args if a in ("clone", "add", "diff", "commit", "push")), args[0])
                raise RuntimeError("Git {} failed (exit {}):\n{}".format(
                    operation, error.returncode, detail[-2000:])) from None
            finally:
                if descriptor is not None:
                    os.close(descriptor)

        root = Path(os.fsdecode(git("rev-parse", "--show-toplevel")).strip())
        if not (root / "thesis.tex").is_file():
            raise RuntimeError("Open this notebook from the thesis repository.")
        files = set(git("ls-files", "--cached", "--others",
                        "--exclude-standard", "-z", cwd=root).split(b"\0"))
        with warnings.catch_warnings():
            warnings.simplefilter("error", getpass.GetPassWarning)
            token = getpass.getpass("Overleaf Git token (hidden): ")
        if not token or len(token.encode("utf-8")) > 4096 or any(c in token for c in "\r\n\0"):
            raise RuntimeError("Enter a valid Overleaf Git token.")

        print("Syncing local files to Overleaf...")
        destination = Path(folder) / "project"
        git("clone", "--quiet", url, str(destination), authenticate=True)
        for item in destination.iterdir():
            if item.name != ".git":
                if item.is_dir() and not item.is_symlink():
                    shutil.rmtree(item)
                else:
                    item.unlink()
        for raw in sorted(files):
            if not raw:
                continue
            relative = Path(os.fsdecode(raw))
            source = root / relative
            if source.is_symlink() or source.is_dir():
                raise RuntimeError("Symlinks and submodules are not supported.")
            if source.is_file():
                target = destination / relative
                target.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(source, target)
        git("add", "--all", "--force", cwd=destination)
        if not git("diff", "--cached", "--name-only", cwd=destination).strip():
            print("Overleaf is already up to date.")
            return
        git("-c", "user.name=Local Sync", "-c", "user.email=sync@localhost",
            "-c", "commit.gpgsign=false", "commit", "--quiet", "-m", "sync",
            cwd=destination)
        git("push", "--quiet", "--no-follow-tags", "origin", "HEAD",
            cwd=destination, authenticate=True)
        print("Success: local files synced to Overleaf.")

try:
    sync_overleaf()
except RuntimeError as error:
    print("Stopped:", error)
except KeyboardInterrupt:
    print("Interrupted. Check Overleaf before retrying.")
except Exception as error:
    print("Sync failed ({}). Check your network and token before retrying."
          .format(type(error).__name__))


Syncing local files to Overleaf...
Sync failed (TimeoutExpired). Check your network and token before retrying.
